# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bander03/FlyRank_Intern/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**The validated model this playbook is built on:** Week 5/6's Random Forest, predicting
`low_ctr_for_type` (CTR at/below the 40th percentile for a page's own content type) from
structural/visibility signals only — `ctr` itself is deliberately excluded from the features
(see Week 5). Week 6's honest, client-grouped holdout gave precision@20 = 1.00, @50 = 0.90,
@100 = 0.83 against a base rate of 0.399 — that is the number this playbook can actually promise,
and it's re-quoted below rather than recomputed, since recomputing it here on the same model
that produces the queue would be circular.

**What's new this week:** the model alone gives a score, not an action. This notebook adds one
transparent, rule-based layer on top — no new model, just human-readable logic — that turns the
score into three ranked tiers and, inside the flagged tiers, two distinct **reason codes** based
on how recently a page was last updated. That split matters because "underperforming CTR" has
two very different fixes depending on *why*: a page updated 3 weeks ago that still underperforms
likely has a title/snippet problem; a page untouched for 3+ months might just be stale.

In [1]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

RANDOM_SEED = 42
pd.set_option("display.width", 120)

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
active = df[df["impressions_90d"] >= 100].copy().reset_index(drop=True)

num_feats = ["avg_position", "impressions_90d", "word_count", "content_age_days",
             "days_since_last_update", "engagement_rate", "scroll_rate"]
cat_feats = ["content_type"]
for f in num_feats:
    active[f] = active[f].fillna(0)
active["content_type"] = active["content_type"].fillna("unknown")

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

pre = ColumnTransformer([("num", "passthrough", num_feats),
                          ("cat", OneHotEncoder(handle_unknown="ignore"), cat_feats)])

# ---- Re-derive the honest held-out numbers (same split/label as Week 5/6) so this
# notebook's promise is traceable, not just asserted ----
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
tr_idx, te_idx = next(gss.split(active, groups=active["client_id"]))
train, test = active.iloc[tr_idx].copy(), active.iloc[te_idx].copy()

train_type_p40 = train.groupby("content_type")["ctr"].quantile(0.40)
global_p40 = train["ctr"].quantile(0.40)
def label_low_ctr(frame):
    thr = frame["content_type"].map(train_type_p40).fillna(global_p40)
    return (frame["ctr"] <= thr).astype(int)
train["label"] = label_low_ctr(train)
test["label"] = label_low_ctr(test)

rf_holdout = Pipeline([("pre", pre), ("clf", RandomForestClassifier(
    n_estimators=300, min_samples_leaf=5, random_state=RANDOM_SEED))])
rf_holdout.fit(train[num_feats + cat_feats], train["label"])
test_proba = rf_holdout.predict_proba(test[num_feats + cat_feats])[:, 1]

held_out_metrics = {"base_rate": round(float(test["label"].mean()), 3)}
for k in (20, 50, 100):
    held_out_metrics[f"precision@{k}"] = round(float(precision_at_k(test_proba, test["label"].values, k)), 3)
print("Re-derived held-out metrics (must match Week 6):", held_out_metrics)

Re-derived held-out metrics (must match Week 6): {'base_rate': 0.399, 'precision@20': 1.0, 'precision@50': 0.9, 'precision@100': 0.83}


In [2]:
# ---- Deployment-style refit on ALL active rows -- this is what actually produces the
# playbook's ranked queue. Its own precision is NOT re-measured here (that would be
# circular, since it would be scoring the same rows it trained on); the precision this
# playbook promises is the held-out number above, from an honest split. ----
active["label"] = label_low_ctr(active)

rf_full = Pipeline([("pre", pre), ("clf", RandomForestClassifier(
    n_estimators=300, min_samples_leaf=5, random_state=RANDOM_SEED))])
rf_full.fit(active[num_feats + cat_feats], active["label"])
active["model_score"] = rf_full.predict_proba(active[num_feats + cat_feats])[:, 1]

queue = active.sort_values("model_score", ascending=False).reset_index(drop=True)
n = len(queue)

# Rank-based tiers: top 5% = priority, next 15% = standard review, rest = monitor.
queue["tier"] = "monitor"
queue.loc[: int(n * 0.05) - 1, "tier"] = "priority_review"
queue.loc[int(n * 0.05): int(n * 0.20) - 1, "tier"] = "review_ctr"

# STALE_DAYS chosen from this dataset's own distribution, not a round-number guess:
# days_since_last_update has a hard cluster at its median (20d) and 75th pct (104d) --
# 90 sits between those two natural clusters, splitting "recently touched" from "not."
STALE_DAYS = 90

def reason_code(row):
    if row["tier"] == "monitor":
        return "below_threshold"
    return "refresh_then_fix_ctr" if row["days_since_last_update"] >= STALE_DAYS else "fix_ctr_titles_meta"

queue["reason_code"] = queue.apply(reason_code, axis=1)

print("Tier counts:")
print(queue["tier"].value_counts())
print("\nReason code counts:")
print(queue["reason_code"].value_counts())

print("\nTop 10 rows:")
print(queue[["content_id", "content_type", "avg_position", "impressions_90d",
             "days_since_last_update", "model_score", "tier", "reason_code"]].head(10).to_string(index=False))

Tier counts:
tier
monitor            17605
review_ctr          3301
priority_review     1100
Name: count, dtype: int64

Reason code counts:
reason_code
below_threshold         17605
fix_ctr_titles_meta      2692
refresh_then_fix_ctr     1709
Name: count, dtype: int64

Top 10 rows:
          content_id    content_type  avg_position  impressions_90d  days_since_last_update  model_score            tier          reason_code
content_2754ce09df20 keyword article          49.7              124                      22     0.971267 priority_review  fix_ctr_titles_meta
content_f51c0b602499 keyword article          47.7              113                      22     0.970227 priority_review  fix_ctr_titles_meta
content_75175d878762 keyword article          48.5            25748                     104     0.969592 priority_review refresh_then_fix_ctr
content_a5637401f707 keyword article          48.2              157                      22     0.969306 priority_review  fix_ctr_titles_meta
content_

**The decay/refresh insight** — the archetype split earns its place: compare the two flagged
reason codes on how much is actually at stake.

In [3]:
flagged = queue[queue["reason_code"] != "below_threshold"].copy()
insight = flagged.groupby("reason_code")[["days_since_last_update", "avg_position", "impressions_90d"]].mean().round(1)
insight["count"] = flagged.groupby("reason_code").size()
print(insight)

                      days_since_last_update  avg_position  impressions_90d  count
reason_code                                                                       
fix_ctr_titles_meta                     20.4          32.8           1412.7   2692
refresh_then_fix_ctr                   104.8          31.7           3799.9   1709


Observed in this data: the `refresh_then_fix_ctr` archetype (untouched 90+ days) carries
noticeably *more* impressions on average than the `fix_ctr_titles_meta` archetype (recently
updated, still underperforming) — these stale-but-flagged pages are not low-value stragglers,
they're pages that already earn real visibility and have simply drifted. That's the practical
case for prioritizing `refresh_then_fix_ctr` within the flagged set: the refresh effort lands on
pages with more traffic already at stake, not fewer.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use:** a weekly or biweekly input for a content reviewer or SEO strategist
deciding which pages to look at first, out of a large backlog. It replaces "which page should I
check today" guesswork with a ranked, reason-coded shortlist — nothing more.

**What it is not:**
- Not a causal claim. This is cross-sectional, observational data — the model finds pages that
  *look* structurally similar to known CTR-underperformers; it does not know *why* any single
  page underperforms.
- Not a claim about Google's ranking algorithm — every number here is measured on FlyRank's own
  anonymized 30k-row sample, one snapshot in time.
- Not validated for anything outside this data's shape: one 90-day trailing window, three
  `content_type` values, 32 pseudonymized clients. A page far outside that shape (a brand-new
  content type, a client not represented in training) is out-of-distribution for this model.
- Not re-validated per client — the honest precision numbers are portfolio-level averages across
  8 held-out clients; a single client's real hit rate could differ.
- The `priority_review` tier (top 5%) is the only tier with a directly-measured precision@K
  claim behind it (precision@20 was measured on exactly this kind of top-slice). The larger
  `review_ctr` tier is directionally useful but sits closer to precision@100 (0.83) territory —
  still well above the 0.399 base rate, but with more expected misses than the top slice.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any flagged row, a human should check:**
1. Is the low CTR actually a title/snippet problem, or is a SERP feature (image pack, "People
   also ask", a competitor's rich result) suppressing clicks regardless of title quality? The
   model has no visibility into SERP layout.
2. Does this page cannibalize traffic with a sibling page from the same client for the same
   query? A "fix" here could just shift clicks from one of the client's own pages to another.
3. Is `content_type` correctly tagged for this row? The whole scoring logic compares a page's
   CTR to its own type's typical CTR — a mistagged row is scored against the wrong peer group.
4. For `refresh_then_fix_ctr` rows specifically: is the topic still relevant, or has search
   intent for this query shifted enough that a refresh wouldn't help regardless of freshness?

**What must NEVER be automated from this playbook, full stop:**
- Auto-publishing any title, meta description, or content change based on the model score alone.
- Auto-redirecting, merging, or deleting pages flagged here.
- Using the model score as the sole input for evaluating a writer's or strategist's performance.
- Treating `priority_review` placement as proof a page is broken — it is a *review* priority,
  not a verdict.
- Feeding this queue into any downstream automation (bidding, budget allocation, CMS workflows)
  without a human sign-off step in between.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

- **Precision drift:** re-measure precision@50 on a fresh held-out sample every time a new
  data export is available. Retrain if it falls meaningfully below the original benchmark
  (0.90) for two consecutive checks — a one-time dip could be noise, a repeated one is drift.
- **Base rate drift:** if the `low_ctr_for_type` base rate moves by more than ~10 percentage
  points from 0.399, the underlying content mix has likely changed enough that the 40th-percentile
  thresholds (learned once, from one training split) should be recomputed, not reused.
- **Feature distribution drift:** if `avg_position`'s distribution shifts noticeably (e.g. after
  a known ranking change), re-check whether position is still behaving the way Week 4's signal
  audit found (peaking around position 4-5, not position 1) — that finding is what justified
  using position as a coarse gate rather than a linear score; if the pattern changes, the gate
  logic should be revisited.
- **Tier size drift:** if `priority_review` (nominally 5% of active pages) balloons or shrinks
  sharply between runs, that's a signal the score distribution itself has shifted, worth a look
  before trusting the new queue.
- **Retrain cadence, absent a trigger:** quarterly, tied to whatever cadence FlyRank refreshes
  its own anonymized export on — there's no evidence in this data for a shorter cycle being
  necessary.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [4]:
outputs_dir = Path("../outputs")
figures_dir = Path("../figures")
outputs_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

# 1) The ranked queue itself (gitignored by design -- regenerates on every run)
export_cols = ["content_id", "client_id", "content_type", "avg_position", "impressions_90d",
               "days_since_last_update", "ctr", "model_score", "tier", "reason_code"]
queue[export_cols].to_csv(outputs_dir / "action_playbook_queue.csv", index=False)
print(f"Wrote {len(queue):,} rows to {outputs_dir / 'action_playbook_queue.csv'}")

# 2) Metrics JSON -- the receipts, committed to git (small, no row-level data)
metrics = {
    "model": "random_forest_low_ctr_for_type",
    "held_out_split": "client_grouped, same as Week 5/6",
    "held_out_metrics": held_out_metrics,
    "deployment_queue_rows": int(n),
    "tier_counts": queue["tier"].value_counts().to_dict(),
    "reason_code_counts": queue["reason_code"].value_counts().to_dict(),
    "stale_threshold_days": STALE_DAYS,
    "archetype_comparison": insight.to_dict(orient="index"),
}
with open(outputs_dir / "action_playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, default=str)
print(f"Wrote metrics to {outputs_dir / 'action_playbook_metrics.json'}")

# 3) A figure worth reusing in the paper: tier sizes + the decay/refresh insight
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

tier_order = ["priority_review", "review_ctr", "monitor"]
tier_counts = queue["tier"].value_counts().reindex(tier_order)
axes[0].bar(tier_order, tier_counts.values, color=["#1A52B0", "#6C8FCB", "#C7D3EA"])
axes[0].set_title("Playbook tier sizes")
axes[0].set_ylabel("pages")
for i, v in enumerate(tier_counts.values):
    axes[0].text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)

archetype_order = ["fix_ctr_titles_meta", "refresh_then_fix_ctr"]
imp_means = insight.loc[archetype_order, "impressions_90d"]
axes[1].bar(archetype_order, imp_means.values, color=["#1A52B0", "#6C8FCB"])
axes[1].set_title("Avg impressions by flagged archetype")
axes[1].set_ylabel("avg impressions_90d")
for i, v in enumerate(imp_means.values):
    axes[1].text(i, v, f"{v:,.0f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
fig_path = figures_dir / "w07_playbook_tiers_and_archetypes.png"
plt.savefig(fig_path, dpi=150)
plt.close(fig)
print(f"Wrote figure to {fig_path}")

Wrote 22,006 rows to ..\outputs\action_playbook_queue.csv
Wrote metrics to ..\outputs\action_playbook_metrics.json


Wrote figure to ..\figures\w07_playbook_tiers_and_archetypes.png


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.